# Part 1: Dataset Exploration and Preparation

We explore the RealWaste images, the waste descriptions and the policy documents, then build the pipelines and the train / validation / test splits.

Files needed next to this notebook: the folder `RealWaste` (one sub-folder per class), `waste_descriptions.csv` and `waste_policy_documents.json`. Numbers in the Results come from my runs; a rerun can differ slightly where sampling is involved.

## Step 1: Loading Libraries

In [ ]:
import os
import re
import gc
import json
import random
import hashlib
import pathlib
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# Section A: The RealWaste Images

## Step 2: Images per Class

In [ ]:
data_dir = pathlib.Path("RealWaste")
class_names = sorted([p.name for p in data_dir.glob("*") if p.is_dir()])
print("classes:", len(class_names))
print(class_names)

counts = {}
for name in class_names:
    counts[name] = len(list((data_dir / name).glob("*.jpg")))
img_counts = pd.Series(counts).sort_values(ascending=False)
print(img_counts)

img_counts.plot.barh(title="Images per class")
plt.show()

In [ ]:
print("largest / smallest:", img_counts.max() / img_counts.min())
print((img_counts / img_counts.sum()).round(3))
print("total images:", img_counts.sum())

**Result:** 9 class folders matching the 9 text labels. Plastic (921) and Metal (790) are the largest, Textile Trash (318) the smallest, a ratio of about 2.90 (shares 0.194 down to 0.067). The images are far less balanced than the text, so class weights and a stratified split are needed later.

## Step 3: Image Size and Colour Mode

We open 40 random images per class (360 in total) and record their size and mode.

In [ ]:
rows = []
for name in class_names:
    files = sorted((data_dir / name).glob("*.jpg"))
    for f in random.sample(files, 40):
        with Image.open(f) as img:
            rows.append((name, img.width, img.height, img.mode))

size_table = pd.DataFrame(rows, columns=["class", "width", "height", "mode"])
print(size_table.groupby(["width", "height"]).size())
print(size_table["mode"].value_counts())

**Result:** All 360 sampled images are 524 x 524 RGB. They are square, so resizing to 224 x 224 does not distort them. This is a sample, so uniform size is assumed for the rest.

## Step 4: Look at the Images

Four random images per class.

In [ ]:
fig, axes = plt.subplots(len(class_names), 4, figsize=(10, 2.3 * len(class_names)))
for row, name in enumerate(class_names):
    files = sorted((data_dir / name).glob("*.jpg"))
    for col, f in enumerate(random.sample(files, 4)):
        axes[row, col].imshow(Image.open(f))
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

**Result:** Top-down photos on a light grey surface, so the background does not give the class away. Lighting varies (some bluish images). Food Organics and Vegetation show speckled piles. Look-alike pairs: Paper and Cardboard, Plastic and Miscellaneous Trash, Food Organics and Vegetation. Miscellaneous Trash has no consistent look. Based on only 36 images.

## Step 5: Brightness and Sharpness

Sharpness is the variance of a simple edge filter: low means blurry. 40 images per class.

In [ ]:
def sharpness(gray):
    g = gray.astype("float32")
    edges = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]
    return float(edges.var())

rows = []
for name in class_names:
    files = sorted((data_dir / name).glob("*.jpg"))
    for f in random.sample(files, 40):
        with Image.open(f) as img:
            gray = np.asarray(img.convert("L").resize((256, 256)))
        rows.append((name, float(gray.mean()), sharpness(gray)))
quality = pd.DataFrame(rows, columns=["class", "brightness", "sharpness"])

In [ ]:
def strip_plot(column, title, xlabel):
    plt.figure(figsize=(8, 5))
    for i, name in enumerate(class_names):
        values = quality[quality["class"] == name][column]
        jitter = np.random.uniform(-0.2, 0.2, len(values))
        plt.scatter(values, i + jitter, s=14, alpha=0.5, color="#2a6f97")
        plt.scatter(values.mean(), i, s=90, color="#e63946", zorder=3)
    plt.yticks(range(len(class_names)), class_names)
    plt.xlabel(xlabel)
    plt.title(title + " (blue = one image, red = class average)")
    plt.tight_layout()
    plt.show()

strip_plot("brightness", "Brightness by class", "Average brightness (0 dark, 255 bright)")
strip_plot("sharpness", "Sharpness by class", "Sharpness score (low = blurry)")

**Result:** Average brightness is similar across classes (about 150 to 162). Vegetation (about 3150) and Food Organics (about 2100) score far sharper than the rest (about 1000); Glass is lowest (about 770). The score responds to texture as well as focus, so the high scores fit their speckled look. A model could use texture as a shortcut.

## Step 6: Exact Duplicate Images

A file fingerprint (MD5) finds byte-identical copies, which could leak between train and test.

In [ ]:
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hashes = []
for name in class_names:
    for f in (data_dir / name).glob("*.jpg"):
        hashes.append(file_hash(f))

print("images checked:", len(hashes))
print("exact duplicate files:", len(hashes) - len(set(hashes)))

**Result:** 4752 images checked, 0 exact duplicates. Near-duplicates (the same item photographed twice) would not be caught.

# Section B: The Waste Descriptions

## Step 7: Load and Check the Descriptions

In [ ]:
desc = pd.read_csv("waste_descriptions.csv")
print(desc.shape)
desc.info()
print(desc.isna().sum())
desc.head()

**Result:** 5000 rows, 5 text columns. `description` is the input and `category` the target. Only `common_confusion` has missing values (2504).

## Step 8: Duplicates and Conflicting Labels

In [ ]:
print("duplicate rows:", desc.duplicated().sum())
print("duplicate descriptions:", desc["description"].duplicated().sum())
labels_per_description = desc.groupby("description")["category"].nunique()
print("descriptions with more than one category:", (labels_per_description > 1).sum())

**Result:** 7 duplicate rows, 61 duplicate descriptions, 0 conflicting labels. An identical description in train and test would inflate the test score, so duplicates are removed before splitting.

## Step 9: Class Balance

In [ ]:
counts = desc["category"].value_counts()
print(counts)
counts.plot.barh(title="Rows per category")
plt.show()

print("largest / smallest:", counts.max() / counts.min())
print((counts / counts.sum()).round(3))

**Result:** Vegetation is largest (600), Paper smallest (506); shares 0.120 to 0.101, ratio 1.19. A model that always guessed Vegetation would score about 12%. The classes are near-balanced, so no class weights are needed, only a stratified split.

## Step 10: Missing Values in common_confusion

In [ ]:
print(desc.isna().mean())

desc["confusion_missing"] = desc["common_confusion"].isna()
print(desc.groupby("category")["confusion_missing"].mean().round(3))
print(desc.groupby("category")["common_confusion"].nunique())

**Result:** 50.08% of `common_confusion` is missing, spread evenly across categories (0.475 to 0.536). Each category has exactly one distinct note, so the blanks are gaps in a category-level note, not real absences.

## Step 11: Label Leakage

In [ ]:
print(desc.groupby("category")["material_composition"].nunique())
print(desc.groupby("category")["disposal_instruction"].nunique())

**Result:** `material_composition` has 1 value per category and `disposal_instruction` has 5, so with `common_confusion` they contain the answer. Only `description` is used as input; the other columns are kept for the RAG documents.

## Step 12: Description Length

In [ ]:
desc["n_words"] = desc["description"].str.split().str.len()
word_counts = desc["n_words"].value_counts().sort_index()

plt.figure(figsize=(8, 4.5))
bars = plt.bar(word_counts.index, word_counts.values, color="#2a6f97")
plt.bar_label(bars, padding=2, fontsize=9)
plt.xticks(word_counts.index)
plt.xlabel("Words per description")
plt.ylabel("Number of descriptions")
plt.show()

print(desc[desc["n_words"] <= 2].sample(8, random_state=1)[["description", "category"]])

In [ ]:
desc.boxplot(column="n_words", by="category", vert=False, figsize=(8, 5))
plt.suptitle("")
plt.title("Words per description by category")
plt.show()

**Result:** Lengths run from 1 word (21 rows) to 10 (13 rows); most are 4 or 5 words. One- and two-word descriptions ("vegetable waste", "pizza box") give a model very little to go on. Paper is slightly shorter than the others, but the ranges overlap heavily.

## Step 13: Vocabulary and Top Words

In [ ]:
print(desc.sample(10, random_state=1)[["description", "category"]].to_string())

words = desc["description"].str.lower().str.findall(r"[a-z]+")
vocab = collections.Counter()
for word_list in words:
    vocab.update(word_list)
print("vocabulary size:", len(vocab))
print(vocab.most_common(15))

In [ ]:
for category in sorted(desc["category"].unique()):
    category_words = collections.Counter()
    for word_list in words[desc["category"] == category]:
        category_words.update(word_list)
    print(category, "->", [w for w, _ in category_words.most_common(6)])

**Result:** The descriptions are template-built (condition word, size or colour word, item, ending), with a vocabulary of only 309 words. Item words name the category (box, peel, can). Template words (with, sized) appear everywhere. "bottle" is a top word for both Glass and Plastic.

## Step 14: Misleading Material Words

In [ ]:
lower = desc["description"].str.lower()
rows = []
for word in ["glass", "metal", "plastic", "paper", "cardboard"]:
    has_word = lower.str.contains(rf"\b{word}\b")
    same = has_word & (desc["category"].str.lower() == word)
    other = int((has_word & ~same).sum())
    rows.append({"word": word, "rows": int(has_word.sum()), "same class": int(same.sum()),
                 "other class": other, "share other": round(other / has_word.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Result:** A material word often sits in a row of another class ("empty patterned glass old t-shirt" is Textile Trash). Share in another class: cardboard 0.254, plastic 0.239, paper 0.218, metal 0.207, glass 0.101. A keyword model would be misled.

## Step 15: After Removing Duplicates

In [ ]:
desc_unique = desc.drop_duplicates(subset="description")
print(len(desc), len(desc_unique))
print(desc_unique["category"].value_counts(normalize=True).round(3))

**Result:** 5000 rows become 4939 and the class shares barely move (Paper 0.101 to 0.100), so the near-balance holds.

# Section C: The Policy Documents

## Step 16: Load the Policy Documents

In [ ]:
policies = json.load(open("waste_policy_documents.json"))
pol = pd.DataFrame(policies)
pol["n_words"] = pol["document_text"].str.split().str.len()
print(len(policies), "documents")
print(pol[["policy_id", "policy_type", "categories_covered", "n_words"]].to_string(index=False))
print(pol["jurisdiction"].value_counts())

**Result:** 14 documents, all for "Metro City", 86 to 146 words each. Docs 1 to 9 cover one category; docs 10 to 14 cover several.

## Step 17: Document Structure

In [ ]:
header_pattern = re.compile(r"^([A-Za-z][A-Za-z \-/&]+):\s*$", re.M)
for _, row in pol.iterrows():
    print(row["policy_id"], header_pattern.findall(row["document_text"]))

**Result:** Docs 1 to 8 share five headers (Acceptable Items, Non-Acceptable Items, Collection Method, Preparation Instructions, Benefits). Doc 9 (Miscellaneous Trash) has its own layout with no "Acceptable Items". Docs 10 to 14 start with GENERAL REQUIREMENTS and then list each category with no acceptable / non-acceptable split.

## Step 18: Category Coverage

In [ ]:
coverage = collections.Counter()
for categories in pol["categories_covered"]:
    for c in categories:
        coverage[c] += 1
print(pd.Series(coverage).sort_values())

**Result:** Paper appears in 1 document, Food Organics in 2, Glass, Textile Trash, Plastic and Cardboard in 3, Vegetation, Metal and Miscellaneous Trash in 4. Paper has no second source to fall back on.

## Step 19: Repeated Content

In [ ]:
bullets = []
for text in pol["document_text"]:
    for line in text.splitlines():
        if line.startswith("- "):
            bullets.append(line[2:].strip())
repeats = pd.Series(bullets).value_counts()
print(repeats[repeats > 1].head(15))

**Result:** The four GENERAL REQUIREMENTS bullets appear 5 times (once in each of docs 10 to 14), and the Vegetation, Metal and Miscellaneous Trash lists 4 times. The multi-category documents mostly recombine the same lists, so indexing all of them would return near-identical chunks.

## Step 20: Window Glass: Do the Documents Agree?

In [ ]:
def split_sections(text):
    parts = header_pattern.split(text)
    headers = parts[1::2]
    bodies = [b.strip() for b in parts[2::2]]
    return list(zip(headers, bodies))

for _, row in pol.iterrows():
    for header, body in split_sections(row["document_text"]):
        for line in body.splitlines():
            if "window glass" in line.lower():
                print("doc", row["policy_id"], "|", header, "|", line.strip())

**Result:** "Window glass or mirrors" appears in docs 2, 11 and 13. In doc 2 it is under Non-Acceptable Items; in docs 11 and 13 it sits in the GLASS GUIDELINES list, which reads as accepted. The documents disagree, and doc 2 (the category-specific one) agrees with the CSV note.

## Step 21: Policy Language Compared with the Descriptions

In [ ]:
policy_words = set(re.findall(r"[a-z]+", " ".join(pol["document_text"]).lower()))
shared = [w for w in vocab if w in policy_words]
covered = sum(vocab[w] for w in shared) / sum(vocab.values())

print("policy vocabulary:", len(policy_words))
print("description vocabulary:", len(vocab))
print("description words also in the policy text:", len(shared))
print("share of word occurrences covered:", round(covered, 3))

**Result:** Only 69 of the 309 description words appear in the policy text (about 28% of word occurrences, my run). Matching a raw description against policy text would work badly, so the plan is to classify first and then retrieve by category.

# Section D: Pipelines and Splits

## Step 22: Image Pipeline (provided)

The provided code resizes to 224 x 224 and keeps 80% for training and 20% as a hold-out, then cuts the hold-out in half. I removed `.cache()` to save memory.

In [ ]:
BATCH_SIZE = 32
IMG_HEIGHT = 224
IMG_WIDTH = 224

num_classes = len(class_names)
print("Number of classes:", num_classes)
print("Total images found:", len(list(data_dir.glob("*/*.jpg"))))

train_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

validation_ds = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

val_batches = tf.data.experimental.cardinality(validation_ds)
test_dataset = validation_ds.take(val_batches // 2)
validation_ds = validation_ds.skip(val_batches // 2)

print("training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("validation batches:", tf.data.experimental.cardinality(validation_ds).numpy())
print("test batches:", tf.data.experimental.cardinality(test_dataset).numpy())

**Result:** 4752 images in 9 classes: 3802 for training and 950 held out, giving 119 training batches and 15 each for validation and test.

## Step 23: Check the Split for Overlap

The provided code cuts the shuffled hold-out with `take` and `skip`. We fingerprint the pixels of validation and test to see whether any image is in both.

In [ ]:
def image_hashes(dataset):
    found = []
    for images, labels in dataset:
        for image in images.numpy():
            found.append(hashlib.md5(image.tobytes()).hexdigest())
    return found

test_hashes = image_hashes(test_dataset)
val_hashes = image_hashes(validation_ds)
print("images read (validation, test):", len(val_hashes), len(test_hashes))
print("images in both:", len(set(test_hashes) & set(val_hashes)))

**Result:** 129 images appear in both validation and test (a first, fuller check found none shared with training). The shuffled hold-out is read twice, once by `take` and once by `skip`, in two different orders. A test score on this split would be inflated.

## Step 24: Fix the Split

Read the 950 hold-out images once, keeping each image together with its own label, then split by position with a stratified split.

In [ ]:
full_holdout = keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode="categorical", shuffle=True)

image_list, label_list = [], []
for images, labels in full_holdout:          # one pass: images and labels stay paired
    image_list.append(images.numpy())
    label_list.append(labels.numpy())
all_images = np.concatenate(image_list)
all_labels = np.concatenate(label_list)
label_numbers = all_labels.argmax(axis=1)

val_idx, test_idx = train_test_split(np.arange(len(label_numbers)), test_size=0.5,
                                     stratify=label_numbers, random_state=42)

validation_ds = tf.data.Dataset.from_tensor_slices((all_images[val_idx], all_labels[val_idx])).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((all_images[test_idx], all_labels[test_idx])).batch(BATCH_SIZE)

print("validation per class:", np.bincount(label_numbers[val_idx]))
print("test per class:      ", np.bincount(label_numbers[test_idx]))

del all_images, all_labels, image_list, label_list, full_holdout
_ = gc.collect()

In [ ]:
test_hashes = image_hashes(test_dataset)
val_hashes = image_hashes(validation_ds)
print("images read (validation, test):", len(val_hashes), len(test_hashes))
print("images in both:", len(set(test_hashes) & set(val_hashes)))

**Result:** The split is now 3802 train, 475 validation and 475 test, with 0 images in both validation and test. Class counts are almost identical in validation and test. My first version read images and labels in two separate passes over the shuffled data, so they were not paired; it showed up in Part 2 as validation accuracy near 0.13. Textile Trash has only 37 and 38 images, so its scores will be noisy.

## Step 25: Text Pipeline and Split

Clean the descriptions, drop duplicates first, turn categories into numbers, then split 80/10/10 with a stratified split.

In [ ]:
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_data = desc[["description", "category"]].copy()
text_data["clean"] = text_data["description"].apply(clean_text)
text_data = text_data.drop_duplicates(subset="clean").reset_index(drop=True)

LABELS = sorted(text_data["category"].unique())
label_to_id = {label: i for i, label in enumerate(LABELS)}
text_data["y"] = text_data["category"].map(label_to_id)

train_data, hold_data = train_test_split(text_data, test_size=0.20, stratify=text_data["y"], random_state=42)
val_data, test_data = train_test_split(hold_data, test_size=0.50, stratify=hold_data["y"], random_state=42)

print("rows after removing duplicates:", len(text_data))
print("train / validation / test:", len(train_data), len(val_data), len(test_data))
print(label_to_id)

# no description may appear in two sets
assert not (set(train_data["clean"]) & set(val_data["clean"]))
assert not (set(train_data["clean"]) & set(test_data["clean"]))
assert not (set(val_data["clean"]) & set(test_data["clean"]))
print("no description appears in more than one set")

**Result:** 4939 rows after removing duplicates; 3951 train, 494 validation, 494 test. Only `description` is used as input. Hyphens are kept and stop-words are not removed, because the descriptions are short and "with" is part of the template.

## Step 26: Documents for the RAG Model

Split each policy document by section and tag every chunk with its category.

In [ ]:
chunks = []
for _, row in pol.iterrows():
    for header, body in split_sections(row["document_text"]):
        categories = row["categories_covered"]
        match = re.match(r"([A-Z ]+) GUIDELINES$", header)
        if len(categories) > 1:
            if header == "GENERAL REQUIREMENTS":
                categories = ["ALL"]
            elif match:
                own = [c for c in categories if c.upper() == match.group(1).strip()]
                categories = own or categories
        chunks.append({"chunk_id": f"policy{row['policy_id']}::{header}", "policy_id": row["policy_id"],
                       "header": header, "category": categories[0],
                       "text": f"{row['policy_type']} | {header}\n{body}"})
chunk_table = pd.DataFrame(chunks)
print("chunks:", len(chunk_table))
print(chunk_table[chunk_table["policy_id"] >= 10][["chunk_id", "category"]].to_string(index=False))

In [ ]:
# the multi-category sections add nothing: every bullet is already in the category's own document
def bullets_of(text):
    return {line[2:].strip() for line in text.splitlines() if line.startswith("- ")}

own_docs = chunk_table[chunk_table["policy_id"] <= 9]
new_bullets = 0
for _, row in chunk_table[(chunk_table["policy_id"] >= 10) & (chunk_table["category"] != "ALL")].iterrows():
    known = set()
    for text in own_docs[own_docs["category"] == row["category"]]["text"]:
        known |= bullets_of(text)
    new_bullets += len(bullets_of(row["text"]) - known)
print("bullets in docs 10-14 not found in the category's own document:", new_bullets)

general = chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"]["text"].str.split("\n", n=1).str[1]
print("distinct GENERAL REQUIREMENTS texts:", general.nunique())

In [ ]:
# final corpus: category documents + one general chunk + one disposal card per category
policy_chunks = pd.concat([own_docs, chunk_table[chunk_table["header"] == "GENERAL REQUIREMENTS"].head(1)])

cards = []
for category, group in desc.groupby("category"):
    instructions = sorted(group["disposal_instruction"].dropna().unique())
    note = group["common_confusion"].dropna().unique()
    text = (f"{category} | Disposal card\nComposition: {group['material_composition'].iloc[0]}\n"
            f"Disposal: " + " ".join(instructions))
    if len(note) > 0:
        text += "\nCommon confusion: " + note[0]
    cards.append({"chunk_id": f"card::{category}", "policy_id": None, "header": "Disposal card",
                  "category": category, "text": text})

rag_corpus = pd.concat([policy_chunks, pd.DataFrame(cards)], ignore_index=True)
print("chunks:", len(rag_corpus))
print(rag_corpus["category"].value_counts().to_dict())

**Result:** 67 chunks, of which 54 are kept: 45 policy chunks and 9 disposal cards. No bullet in docs 10 to 14 is missing from the category documents and the 5 general-requirements copies are identical, so the repeated sections are dropped. This also removes the conflicting window-glass placement. Every category has 6 chunks (Miscellaneous Trash 5). Part 4 rebuilds this corpus.

## Summary

- **Images:** 4752 images, 9 classes, uneven (ratio 2.90), 524 x 524 RGB, no exact duplicates. The provided split leaked (129 images in both validation and test); the fixed split is 3802 / 475 / 475, stratified.
- **Descriptions:** 5000 rows (4939 after duplicates), template-built with a 309-word vocabulary. Three columns leak the label, so only `description` is used. Split 3951 / 494 / 494.
- **Policy documents:** 14 documents, uneven coverage, one real contradiction (window glass). The RAG corpus has 54 chunks.
- **Limitations:** size, quality and grid findings come from samples; near-duplicate and mislabelled images were not checked; Textile Trash has few test images; the text is template data, so scores on it flatter a model.